# 🟢 cash — live feature tour

[cash](https://github.com/galgtonold/cash) caches your notebook **one statement at a time** and tracks how cells depend on each other, so re-running only recomputes what actually changed — nothing else.

This tour runs a small model bake-off — one shared feature stage, three models competing on the same data, one leaderboard — and shows what cash does for you along the way. The whole thing runs in well under a minute.

**How to use this notebook**
1. **Run all.** Every cell runs once and prints a cash **badge** summarizing what it did (`EXECUTED`, in ochre).
2. **Run all a _second_ time.** The expensive cells finish in hundredths of a second, each badge carrying a green **`cached`** chip and a **`saved`** figure — the slow work is restored from RAM, not repeated.
3. **Edit `BOOST_ROUNDS`** (§3) and Run all — only the boosting model pays again; the other two models and the shared feature stage stay cached.
4. **Edit the shared setting `N_CLUSTERS`** (§4) and run *just the champion cell below it* — cash re-derives the features and all three fits for you.
5. **Edit `score_fit`** (§5) — all three models refit, because all three call it. The k-means stage does not call it, and stays cached.
6. **Restart the kernel** and Run all — the cache survives, so the slow steps still restore.

> ### ⚠️ Save the notebook before you run
>
> Step 4 edits one cell and then runs a **different** one — the champion cell, not the one you just edited. cash reads a cell it didn't execute from the **saved `.ipynb` file**, so an edit still sitting unsaved there is invisible to it: it reads the old value, concludes nothing upstream changed, and hands you the previous answer while your screen shows the new code.
>
> **In JupyterLab (this Binder) and VS Code: press `Ctrl+S` / `Cmd+S` after editing, then run.** Autosave exists, but it runs on a timer that is usually slower than you are. On Google Colab there is nothing to save — cash reads cells live from the frontend.
>
> Step 5 also edits a cell, but its instruction is **Run all**, which re-executes that very cell — so there is nothing unsaved for cash to miss there. Save first anyway, out of habit; it costs nothing.


In [ ]:
# cash is already installed in this Binder environment (see binder/requirements.txt),
# so there's nothing to install here — just import and switch it on.
import cash
import numpy as np
import pandas as pd

%cash_on

## 1 · A bake-off that caches itself

Three models compete to predict next-quarter customer spend, and they all draw
on the same data: one shared feature stage feeds all three, and a leaderboard
ranks them at the end. Every model reports its error through a single scoring
helper. That makes this notebook a lattice — one shared root, three siblings,
one leaf — and that shape is what the rest of the tour plays with.

The next three cells set it up: the scoring helper, the five functions that do
the work, and the panel itself. Building the panel is quick. It is a couple of
megabytes rather than a couple of gigabytes, and it is **seeded**, so the data
is identical every run. The raw data is not the expensive part here.

**The modelling is.** Each fit is slow because it iterates, not because the
data is large — the opposite of the case people usually picture when they think
about caching, a huge table and a trivial operation. It is also the case where
caching pays, because **cash stores the answer, not the work.** A fit that grinds for seconds and hands back a single number is almost free to cache. A fast operation with an enormous result is the opposite, and the badge's `overhead` row will tell you when you have one.

In [ ]:
# Every model below reports its error through this one helper.
# §5 edits it — and all three cached scores notice.
def score_fit(pred, actual):
    return float(np.sqrt(np.mean((pred - actual) ** 2)))

In [ ]:
# The bake-off, in five functions. Everything here is plain numpy — no
# sklearn — so you can read exactly what is being cached and what it costs.
# Each fit is slow because it ITERATES, not because the data is large.

def build_panel(n_customers=12_000, seed=0):
    """A seeded customer panel: 24 features, plus next-quarter spend."""
    rng = np.random.default_rng(seed)
    loading = rng.standard_normal((3, 20))
    latent = rng.standard_normal((n_customers, 3))
    observed = latent @ loading + 0.6 * rng.standard_normal((n_customers, 20))
    tenure = rng.integers(1, 60, n_customers) / 60.0
    recency = rng.integers(0, 90, n_customers) / 90.0
    frequency = np.log1p(rng.poisson(6, n_customers) + 1)
    features = np.column_stack(
        [observed, tenure, recency, frequency, frequency * tenure])
    # A linear part a lasso can nail, plus a saturating term and an
    # interaction only a nonlinear learner can find. Nobody wins by default.
    linear = 24.0 * latent[:, 0] - 13.0 * latent[:, 1]
    curved = 26.0 * np.tanh(2.2 * latent[:, 2]) + 30.0 * (recency < 0.35) * tenure
    spend = 120 + linear + curved + 8.0 * rng.standard_normal(n_customers)
    return features, spend


def cluster_distance_features(X, n_clusters, iters=50, restarts=8, seed=7):
    """k-means from scratch; returns each customer's distance to every centre."""
    rng = np.random.default_rng(seed)
    best_centres, best_inertia = None, np.inf
    for _ in range(restarts):
        centres = X[rng.choice(len(X), n_clusters, replace=False)].copy()
        for _ in range(iters):
            d = ((X**2).sum(1)[:, None] - 2 * X @ centres.T
                 + (centres**2).sum(1)[None, :])
            labels = d.argmin(1)
            for j in range(n_clusters):
                members = labels == j
                if members.any():
                    centres[j] = X[members].mean(0)
        inertia = d[np.arange(len(X)), labels].sum()
        if inertia < best_inertia:
            best_inertia, best_centres = inertia, centres.copy()
    d = ((X**2).sum(1)[:, None] - 2 * X @ best_centres.T
         + (best_centres**2).sum(1)[None, :])
    return np.sqrt(np.maximum(d, 0.0))


def fit_lasso(X, y, alpha, sweeps=250):
    """Lasso by coordinate descent — one feature at a time, many times over."""
    Z = (X - X.mean(0)) / (X.std(0) + 1e-9)
    intercept = y.mean()
    w = np.zeros(Z.shape[1])
    resid = y - intercept
    n = len(Z)
    for _ in range(sweeps):
        for j in range(Z.shape[1]):
            resid += Z[:, j] * w[j]
            rho = Z[:, j] @ resid / n
            w[j] = np.sign(rho) * max(abs(rho) - alpha, 0.0)
            resid -= Z[:, j] * w[j]
    return score_fit(Z @ w + intercept, y)


def fit_boosted_stumps(X, y, rounds, cuts=12, lr=0.5):
    """Gradient boosting on one-split trees, searching every feature x cut."""
    pred = np.full(len(X), y.mean())
    resid = y - y.mean()
    grid = np.percentile(X, np.linspace(10, 90, cuts), axis=0)
    for _ in range(rounds):
        best = (np.inf, 0, 0.0, 0.0, 0.0)
        for j in range(X.shape[1]):
            column = X[:, j]
            for cut in grid[:, j]:
                left = column <= cut
                if not left.any() or left.all():
                    continue
                lo, hi = resid[left].mean(), resid[~left].mean()
                sse = (((resid[left] - lo) ** 2).sum()
                       + ((resid[~left] - hi) ** 2).sum())
                if sse < best[0]:
                    best = (sse, j, cut, lo, hi)
        _, j, cut, lo, hi = best
        left = X[:, j] <= cut
        pred[left] += lr * lo
        pred[~left] += lr * hi
        resid[left] -= lr * lo
        resid[~left] -= lr * hi
    return score_fit(pred, y)


def fit_neural_net(X, y, epochs, hidden=32, lr=0.08, seed=3):
    """One hidden layer, full-batch gradient descent, written out by hand."""
    rng = np.random.default_rng(seed)
    Z = (X - X.mean(0)) / (X.std(0) + 1e-9)
    y_mean, y_std = y.mean(), y.std()
    y_scaled = (y - y_mean) / y_std
    W1 = rng.standard_normal((Z.shape[1], hidden)) * 0.3
    b1 = np.zeros(hidden)
    W2 = rng.standard_normal(hidden) * 0.3
    b2 = 0.0
    n = len(Z)
    for _ in range(epochs):
        H = np.tanh(Z @ W1 + b1)
        err = (H @ W2 + b2) - y_scaled
        dH = np.outer(err, W2) * (1 - H**2)
        W1 -= lr * (Z.T @ dH / n)
        b1 -= lr * dH.mean(0)
        W2 -= lr * (H.T @ err / n)
        b2 -= lr * err.mean()
    return score_fit((np.tanh(Z @ W1 + b1) @ W2 + b2) * y_std + y_mean, y)

In [ ]:
panel, spend = build_panel()

print(f"{len(panel):,} customers × {panel.shape[1]} features "
      f"({panel.nbytes / 1e6:.1f} MB)  |  spend spread ±${spend.std():,.0f}")
pd.DataFrame(panel[:, -4:], columns=["tenure", "recency", "frequency",
                                     "freq×tenure"]).head()

The shared stage: k-means on the panel, written out from scratch — eight
restarts of fifty iterations each — giving every customer its distance to every
cluster centre. Those distances are glued onto the panel to make the design
matrix that **every model below is built on**.

This is the one upstream the whole lattice hangs from, and both of the
experiments later on turn on it: in §3 you change a single model and this stage
stays cached; in §4 you change this stage and everything downstream of it goes.

In [ ]:
# ⚙️  Shared upstream setting — every model below is built on these features.
N_CLUSTERS = 24

In [ ]:
# k-means, 8 restarts × 50 iterations. Seconds of genuine CPU, and the result
# is a distance matrix cash hashes in about 2 ms — slow to compute, cheap to
# track. That ratio is what makes caching worth having.
cluster_feats = cluster_distance_features(panel, N_CLUSTERS)
design = np.hstack([panel, cluster_feats])

print(f"design matrix {design.shape}  ({design.nbytes / 1e6:.1f} MB)")

Now the three challengers, each with its own hyperparameter in its own cell
just above it:

- **Lasso**, by coordinate descent — one feature at a time, hundreds of sweeps
  over all of them, tuned by `LASSO_ALPHA`.
- **Gradient-boosted stumps** — every round searches each feature × cut pair
  for the best single split, and there are `BOOST_ROUNDS` rounds of it.
- **A one-hidden-layer neural net** — full-batch gradient descent written out
  by hand, run for `MLP_EPOCHS` epochs.

Each one takes a few seconds, and none of that is data loading; it is all
iteration. All three report their error through the same `score_fit` helper
from the top of the notebook — remember that for §5.

In [ ]:
LASSO_ALPHA = 0.5

In [ ]:
lasso_rmse = fit_lasso(design, spend, LASSO_ALPHA)
print(f"lasso            RMSE {lasso_rmse:6.2f}")

In [ ]:
# 👇  EDIT THIS NUMBER for §3 — try 40. Only the boosting recomputes.
BOOST_ROUNDS = 25

In [ ]:
boost_rmse = fit_boosted_stumps(design, spend, BOOST_ROUNDS)
print(f"boosted stumps   RMSE {boost_rmse:6.2f}")

In [ ]:
MLP_EPOCHS = 250

In [ ]:
mlp_rmse = fit_neural_net(design, spend, MLP_EPOCHS)
print(f"neural net       RMSE {mlp_rmse:6.2f}")

The leaderboard reads all three scores, so it is the cell that shows any of
them changing.

It is worth a look before you move on. Every RMSE here lands well inside the
spend spread printed back in §1, so all three models genuinely learned
something: the target has a linear part a lasso can nail plus a saturating term
and an interaction only a nonlinear learner can find, and nobody wins by
default.

This is also the one cell in the tour that cash refuses to cache, and it says so out loud — a red `not cached` chip, and a reason on each row it names. The `Axes` it returns is identity-coupled to pyplot's own figure state, and the two lines that label it mutate it in place; `plt.show()` is a display side effect besides. So the chart redraws on every run, cold or warm. Once the fits settle to hundredths of a second, this is what is left running — the tour's one recurring cost, not a cheap one.


In [ ]:
import matplotlib.pyplot as plt

leaderboard = pd.Series(
    {"lasso": lasso_rmse, "boosted stumps": boost_rmse, "neural net": mlp_rmse}
).sort_values()

ax = leaderboard[::-1].plot(kind="barh", color="#2e9e6b")
ax.set_title(f"Bake-off — RMSE, lower is better  ({N_CLUSTERS} clusters)")
ax.set_xlabel("RMSE ($)")
plt.tight_layout()
plt.show()

leaderboard.round(2)

## 2 · Run it again — and it's free

Hit **Run all** a second time now, and read the badges on the cells above.

The four expensive cells — the k-means stage and the three fits — finish in
hundredths of a second. Each badge now carries a green **`cached`** chip and a
**`saved`** figure showing the seconds it did not spend. cash recognised that
neither the code nor its inputs had changed, and handed back the stored answers
instead of recomputing them.

Click a badge open and expand the row for the fit: it reads `CACHED`, restored from `RAM` — same kernel, same session, so cash never has to touch disk. The cheap statements beside it — the `print`, mostly — ran as normal, which is also why the headline still says `EXECUTED`. cash caches **per statement**, not per cell, and printing is a side effect it will not skip to give you a tidier badge. What it skipped is the arithmetic, and the arithmetic was all of the time.


## 3 · Change one model — the others stay cached

This is the section the rest of the notebook exists to set up, so give it a
moment.

Scroll up to the cell marked 👇 — `BOOST_ROUNDS = 25` — change the number
(try 40) and hit **Run all**. Then read the badges on the way back down.

The boosting is the only model that pays. Its badge has no green `cached` chip
and nothing `saved`; it spent its seconds over again, because the number it
depends on changed. What did **not** happen is the interesting half:

- the lasso did not refit — green `cached`, a `saved` figure, hundredths of a
  second;
- the neural net did not refit — the same;
- the k-means feature stage was not rebuilt — the same.

One model paid; the other two, and the stage all three share, did not. They are
all built on that stage and all read the same design matrix, so from the outside
they look like a single block of work — and invalidating the lot would have been
the easy answer. cash tracks what each statement actually reads, and worked out
that exactly one of the three depends on the number you changed.

The leaderboard redraws, as it always does, and the boosting's bar visibly
moves in. It does not catch the other two at 40 rounds — and those two sit
close enough to each other that it is worth pushing further.

## 4 · Change the shared stage — then ask for just the answer

§3 ran the lattice one way; this runs it the other. Go back up to the ⚙️ cell
holding `N_CLUSTERS`, change it (try 30), **save the notebook** (`Ctrl+S` /
`Cmd+S`), and then run **only the champion cell below** — select it and press
Ctrl/Cmd+Enter. Don't run the cells in between.

Everything descends from that setting, so everything is stale: the k-means
features, all three fits, the leaderboard. You asked for none of them. cash
re-derives the chain for you and hands you the champion — you changed one
setting and asked one question, and it worked out the minimum it had to redo to
answer you.

> **If the numbers don't change, you skipped the save.** cash reads the cell you
> edited but didn't run from the file on disk, so an unsaved `N_CLUSTERS` is
> still the old `N_CLUSTERS` as far as it can tell. (Not applicable on Colab,
> where cash reads cells live.)

In [ ]:
# 👇  Run ONLY this cell after changing N_CLUSTERS above. cash re-derives the
#     features and all three fits for you — you never run those cells.
champion = leaderboard.idxmin()
print(f"champion: {champion}  (RMSE {leaderboard.min():.2f})  "
      f"with {N_CLUSTERS} cluster features")

## 5 · Edit a helper — everything that calls it updates

cash hashes a function's source **and** the source of the functions it calls.
So when you change an inner helper, every cached result that reaches it —
directly or transitively — invalidates on its own.

`score_fit` is that helper here. It sits in its own cell near the top of the
notebook, and all three fits call it to report their error. Edit it — swap the
RMSE for a mean absolute error, say, `np.mean(np.abs(pred - actual))` in place
of the square root — then **save** (`Ctrl+S` / `Cmd+S`) and **Run all**.

All three models refit, and pay their seconds again. Not one of the three model
cells changed a character; what changed is a function they all call.

Now look at the k-means cell: **it still says `saved`, and still carries its
green `cached` chip.** k-means never calls `score_fit`, so its stored result is
still good, and cash left it alone. That contrast is the whole demonstration —
cash follows the call graph, rather than throwing out everything below the cell
you edited.

## 6 · Beyond notebooks — the `@cash.cache` decorator

Outside cells, wrap any function with `@cash.cache` and it caches by its
arguments and its own source code. The first call runs; an identical call
returns instantly.

Note the `# @cash:assume-safe` on the `time.sleep` line. cash's analyzer flags
calls that look like side effects — a `sleep` throws its return away, which is
exactly the shape of something that matters and would be skipped on a cache
hit. Here it is deliberate, so the line is waived and the warning goes away.

The waiver is **per line**, on purpose. `@cash.cache(assume_safe=True)` would
silence the whole function including anything added to it next year; annotating
the one line you audited means a new unaudited call still speaks up.

In [ ]:
import time

@cash.cache
def spend_percentiles(features, column):
    time.sleep(1.0)  # @cash:assume-safe — the sleep IS the stand-in for slow work
    return np.percentile(features[:, column], [10, 50, 90]).round(3)

print("first call (runs ~1s):")
print(spend_percentiles(panel, 20))
print("\nsecond call (instant, from cache):")
print(spend_percentiles(panel, 20))

## 7 · It survives a kernel restart

The cache lives on disk, not just in memory. Try **restarting the kernel**, then
**Run all**: the slow feature stage and all three fits restore from cache
instead of recomputing — a fresh kernel picks up right where you left off.

## What did cash save you?

Cache hits, misses, and what cash measured this session:

In [ ]:
%cash_stats

### How to read that

Two of those numbers are **measured**, and they are the pair to compare
between your first run and this one:

- **Compute time** — how long your code actually ran. Several seconds on the
  first pass; a fraction of that once the cache is warm.
- **Statements restored** — results handed back instead of recomputed. Zero on
  the first run, and most of the expensive ones afterwards.

**Net time saved** is deliberately the most pessimistic figure cash can
defend. It credits only savings it re-measured *this session* — and on a clean
**Run all** a statement either restores from cache or computes, never both, so
there is usually nothing to re-measure. That is why it can show a negative
lower bound on the very run that saved you the most. The `at best` end credits
each value with what it cost when first cached.

cash would rather understate a win than claim one it cannot prove. Compute time
is the number that needs no such caveat.